# BERTopic 主題模型實戰：從分群到讓 GPT 幫主題取名字
---

## 這堂課你會學到什麼

1. 用 **embedding** 把每篇摘要變成向量
2. 用 **UMAP** 降維、用 **HDBSCAN** 自動分群
3. 用 **BERTopic** 把「分群 + 找關鍵字」組裝成完整的主題模型
4. 用 **Plotly 互動圖** 觀察主題
5. 用 **KeyBERTInspired**、**MaximalMarginalRelevance** 微調關鍵字
6. 設計 **提示詞（prompt）**，讓 **GPT** 為每個主題產生一個易讀的標題，並搞懂 `[DOCUMENTS]`、`[KEYWORDS]` 這兩個佔位符是怎麼被填入的

## 0. 環境準備

```bash
pip install bertopic datasets sentence-transformers umap-learn hdbscan matplotlib pandas openai python-dotenv
```

## 1. 先看整體：BERTopic 在做什麼？

**問題**：手上有幾萬篇論文，沒有人幫你標分類，怎麼知道它們在談什麼？

BERTopic 把工作拆成幾個**可抽換的零件**：

```
 文字（摘要）
    │
    ▼ ① Embedding        SentenceTransformer("thenlper/gte-small")
 向量（每篇一個，384 維）
    │
    ▼ ② 降維             UMAP
 低維向量（這裡是 5 維）
    │
    ▼ ③ 分群             HDBSCAN
 每篇得到一個 cluster 編號（-1 = 離群值）
    │
    ▼ ④ 找關鍵字         c-TF-IDF（BERTopic 內部做）
 每個 cluster 的代表詞 = 一個「主題」
    │
    ▼ ⑤（選用）微調關鍵字  KeyBERTInspired / MaximalMarginalRelevance
 更好讀的關鍵字
    │
    ▼ ⑥（選用）生成標題   OpenAI（GPT）
 每個主題一個簡短的人話標題
```

> 📌 **本節重點**
> - BERTopic 是 **embedding → 降維 → 分群 → 關鍵字 → （微調 / 生成標題）** 的流水線，每個零件都能換。
> - 前半段（①②③）先「手動」做，看清楚每一步；後半段交給 BERTopic 一次組裝。

## 2. 匯入套件

| 套件 | 用途 |
|---|---|
| `datasets` | 從 Hugging Face 載入資料集 |
| `pandas` | 表格處理 |
| `SentenceTransformer` | 把文字變成 embedding |
| `UMAP` / `HDBSCAN` | 降維 / 密度分群 |
| `matplotlib` | 靜態散佈圖 |
| `BERTopic` | 主題模型本體 |
| `deepcopy` | 完整複製資料（第 13 節備份用） |
| `KeyBERTInspired`、`MaximalMarginalRelevance` | 微調關鍵字的「表示模型」 |
| `openai` | **OpenAI 官方的** Python 套件，用來建立 client |
| `bertopic.representation.OpenAI` | **BERTopic 的**表示模型，負責把 GPT 接進主題模型 |
| `load_dotenv` | 從 `.env` 檔讀取 API 金鑰 |

> ⚠️ **容易搞混：兩個都叫 OpenAI**
> - 小寫 `openai` 是官方套件，`openai.OpenAI()` 建立「能呼叫 API 的 client」。
> - 大寫 `OpenAI`（從 `bertopic.representation` 匯入）是 BERTopic 的包裝類別，它**拿 client 去呼叫 GPT**，並處理提示詞、取回標籤。
> 這兩個一起用，各管一件事。

In [ ]:
#.venv312
from datasets import load_dataset
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt
from bertopic import BERTopic
from copy import deepcopy
from bertopic.representation import KeyBERTInspired
from bertopic.representation import MaximalMarginalRelevance
import openai
from bertopic.representation import OpenAI
from dotenv import load_dotenv

## 3. 提示詞模板 `PROMPT`：先設計好要問 GPT 什麼

這是第 15 節要送給 GPT 的提示詞。我們先把它寫在這裡，因為它只是一個**字串模板**。

```
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords:
[KEYWORDS].

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
```

兩個方括號的標記叫**佔位符（placeholder）**：

| 佔位符 | BERTopic 會換成什麼 |
|---|---|
| `[DOCUMENTS]` | 這個主題的**代表文件**（預設 4 篇），每篇一行，前面加 `- ` |
| `[KEYWORDS]` | 這個主題的**關鍵字**，用 `, ` 連起來 |

**重點：你不用自己替換。** BERTopic 的 `OpenAI` 表示模型在每次呼叫 GPT 前，會自動把這兩個標記換成該主題的真實內容。
但這兩個名字是 **BERTopic 規定的**，你不能自己取（寫成 `[DOCS]` 它就不認得，而且不會報錯）。

最後一行 `topic: <short topic label>` 是在**規定 GPT 的回答格式**：BERTopic 取回答時，會把開頭的 `topic: ` 去掉，只留標籤本身。所以格式要寫成 `topic:`（冒號前沒有空格）。

> 補充：程式裡 `PROMPT` 的每一行前面有縮排空格（因為原本寫在類別裡），這些空格也會一起送出去，不影響結果，只是多一點不必要的字元。

In [ ]:
PROMPT = """
        I have a topic that contains the following documents:
        [DOCUMENTS]
        
        The topic is described by the following keywords:
        [KEYWORDS].
        
        Based on the information above, extract a short topic label in the following format:
        topic: <short topic label>
    """

> 📌 **本節重點**
> - 佔位符是「程式約定好的標記」，由**負責替換的工具**決定名字；BERTopic 認得 `[DOCUMENTS]` 與 `[KEYWORDS]`。
> - 提示詞裡**明確寫出輸出格式**，才能讓程式穩定地解析回答。
> - 寫成 `topic : ` 與 `topic: ` 看似只差一個空格，但程式去前綴時只認後者。

## 4. 輔助函式 `topic_difference`：比較「改之前」與「改之後」

這個函式到第 13～15 節才會用到。它把每個主題**前 5 個關鍵字**做成表格。核心是這一行：

```python
"| ".join(list(zip(*original_topics[topic]))[0][:5])
```

假設 `original_topics[topic]` 是 `[("robot", 0.42), ("control", 0.37), ...]`：

| 步驟 | 做了什麼 |
|---|---|
| `*` | 把 list 拆開，一個一個丟給 `zip` |
| `zip(...)` | 把所有「第 1 個元素」放一起、「第 2 個元素」放一起（轉置） |
| `list(...)` | 轉成 list（`zip` 本身只是迭代器） |
| `[0]` | 取第一組＝**所有詞** |
| `[:5]` | 只取前 5 個 |
| `"| ".join(...)` | 用 `"| "` 接起來 |

`df.loc[len(df)] = [...]` 是「在表格最後面新增一列」的常見寫法。

In [ ]:
def topic_difference(model, original_topics, nr_topics=5):
    # original_topics[topic] 是很多 (word, score)
    # * 把它們 unpack
    # zip() 把所有第 1 個元素放一起、所有第 2 個元素放一起
    # ex:zip(
    #     ("robot", 0.42),
    #     ("control", 0.37),
    #     ("trajectory", 0.31)
    # )
    # list後會變成
    # [
    #     ("robot", "control", "trajectory", "motion", "manipulator", "system"),
    #     (0.42, 0.37, 0.31, 0.28, 0.25, 0.20)
    # ]
    # [0] 取第一組，也就是所有 word
    # [:5] 取前5個
    # join() 是把很多 string 接起來 ex: robot| control| trajectory
    df = pd.DataFrame(columns=["topic", "Original", "Updated"])

    for topic in range(nr_topics):
        og_words = "| ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = "| ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

### 🧪 教學小實驗（非原程式）：親手拆解 `zip(*...)`

In [1]:
# 🧪 教學小實驗（非原程式）
demo_topic = [("robot", 0.42), ("control", 0.37), ("trajectory", 0.31),
              ("motion", 0.28), ("manipulator", 0.25), ("system", 0.20)]

print(list(zip(*demo_topic)))               # 兩組 tuple：所有詞 / 所有分數
print(list(zip(*demo_topic))[0])            # [0] 取所有詞
print(list(zip(*demo_topic))[0][:5])        # [:5] 取前 5 個
print("| ".join(list(zip(*demo_topic))[0][:5]))   # join 接起來

[('robot', 'control', 'trajectory', 'motion', 'manipulator', 'system'), (0.42, 0.37, 0.31, 0.28, 0.25, 0.2)]
('robot', 'control', 'trajectory', 'motion', 'manipulator', 'system')
('robot', 'control', 'trajectory', 'motion', 'manipulator')
robot| control| trajectory| motion| manipulator


> 📌 **本節重點**
> - `zip(*list_of_pairs)` = 把「一對一對」轉置成「一欄一欄」。
> - 函式只讀資料、回傳表格，不會改動模型。
> - `nr_topics=5` 表示看主題 0～4，模型至少要有這 5 個主題。

## 5. 載入資料（步驟 1）

- `load_dataset(...)['train']` 取出訓練分割。
- 資料集有兩個欄位：`Titles`（標題，只用來畫圖時顯示）和 `Abstracts`（摘要，**拿去做 embedding 與主題建模**）。
- 原程式把幾行 `print` 註解掉了（看資料集、前 5 筆、欄位名稱）。想探索資料時，可以把它們的 `#` 拿掉。

In [ ]:
dataset = load_dataset("MaartenGr/arxiv_nlp")['train']
# print(dataset)
# print(dataset.to_pandas().head())
# print(dataset.column_names)
titles = dataset['Titles']
abstracts = dataset['Abstracts']

> 📌 **本節重點**
> - 主題建模**不需要標籤**，只需要文字。
> - 拿到新資料集，先 `print` 欄位與前幾筆，是好習慣。

## 6. 第一步：把摘要變成 embedding（步驟 2）

- `thenlper/gte-small` 是小而快的句子嵌入模型，每段文字變成一個 **384 維**向量。
- 意思相近的文字，向量會比較靠近 —— 這是後面能「分群」的原因。
- 這一步最花時間（資料筆數很多，你的輸出顯示共 44949 筆）；有 GPU 會快很多。
- `embeddings.shape` 會是 `(摘要篇數, 384)`。

In [ ]:
#載入embedding模型, `thenlper/gte-small` 是一個小而快的句子嵌入模型，每段文字會變成一個384 維的向量。

embedding_model = SentenceTransformer("thenlper/gte-small")  
embeddings = embedding_model.encode(abstracts, show_progress_bar = True)
# print(embeddings.shape)

> 📌 **本節重點**
> - Embedding 把「文字」變成「數字向量」，電腦才能算相似度、做分群。
> - 嵌入結果存起來，UMAP 與 BERTopic 都重複使用，不必重算。

## 7. 第二步：UMAP 降維（步驟 3）

384 維對分群來說太高（維度太高時，點與點的距離會變得不好區分），所以先壓成 5 維。

| 參數 | 意思 |
|---|---|
| `n_components=5` | 壓到 5 維，給**分群**用 |
| `min_dist=0.0` | 允許點靠得很近，讓同類聚得更緊，有利分群 |
| `metric='cosine'` | 用餘弦相似度（文字 embedding 常用） |
| `random_state=42` | 固定隨機種子，結果可重現 |

scikit-learn 風格的三個方法：`fit()` 只學習；`transform()` 用學好的模型轉換；`fit_transform()` 先學習再立刻轉換同一批資料。

In [ ]:
umap_model = UMAP(
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state = 42,  
)
reduced_embeddings = umap_model.fit_transform(embeddings)
# print(reduced_embeddings.shape)

> 📌 **本節重點**
> - UMAP 的目的：**保留「誰跟誰近」**，同時降維。
> - 分群用 5 維、畫圖用 2 維（第 9 節），**兩者不必一樣**。
> - 固定 `random_state` 能重現結果；代價是 UMAP 通常因此無法平行運算、較慢。

## 8. 第三步：HDBSCAN 分群（步驟 4）

| | K-Means | HDBSCAN |
|---|---|---|
| 要先決定幾群嗎 | 要 | **不用** |
| 處理雜訊 | 每個點都得歸某一群 | 不屬於密集區的點標成 **-1（離群值）** |
| 原理 | 距離中心點最近 | 找**密度高**的區域 |

| 參數 | 意思 |
|---|---|
| `min_cluster_size=50` | 一群至少 50 篇；調大 → 群變少變粗 |
| `metric='euclidean'` | 在降維後的空間用歐氏距離 |
| `cluster_selection_method='eom'` | `eom` 選最穩定、有代表性的群；`leaf` 選最細的子群 |

In [ ]:
#HDBSCAN是基於密度的演算法 可以自由計算分群數量
#eom(Excess of Mass) eom = 選「最穩定、有代表性」的 cluster；leaf = 選「最細的子 cluster」。    
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric='euclidean',
    cluster_selection_method='eom',
).fit(reduced_embeddings)

clusters = hdbscan_model.labels_
# print(len(set(clusters)))

原程式把 `print(len(set(clusters)))` 註解掉了。下面的小實驗說明為什麼要注意這個數字：

In [ ]:
# 🧪 教學小實驗（非原程式）
demo_clusters = [0, 0, 1, -1, 2, -1, 1, 0]

print(set(demo_clusters))                      # 不重複的編號
print(len(set(demo_clusters)))                 # 4：包含 -1
print(len(set(demo_clusters)) - (1 if -1 in demo_clusters else 0))   # 3：真正的 cluster 數

> 📌 **本節重點**
> - 標籤 **`-1` 是離群值（noise）**，不是真正的主題。
> - `len(set(clusters))` 會把 `-1` 也算進去；真正的群數要再減 1。

## 9. 畫圖：把 cluster 看見（步驟 5）

分群結果只是一串數字，畫出來才有感覺。這一段**另外**把 embedding 壓成 2 維（螢幕只能畫平面）：

1. `UMAP(n_components=2, ...)`：重新降到 2 維，**只拿來畫圖**。結果存成 `reduced_embeddings_2`，第 12 節還會用到。
2. 建一個 `DataFrame`，欄位 `x`、`y`（座標）、`title`、`cluster`。
3. 用 `cluster != '-1'` / `== '-1'` 把「有群的點」和「離群值」分開。
4. `plt.scatter(x, y, c=cluster編號, cmap='tab20b')`：**`x`、`y` 決定點的位置，`c` 決定顏色**。同一群的點顏色相同。
5. 離群值畫成灰色、很透明（`alpha=0.05`），不搶焦點。

> 注意：2 維圖只是 384 維資料的「投影」；兩群在圖上重疊，不代表原始空間也重疊。分群是在 **5 維**空間完成的。`tab20b` 只有 20 種顏色，你有上百個群，所以顏色會重複，看「哪裡聚成一塊」即可，不要靠顏色分辨每個群。

In [ ]:
reduced_embeddings_2 = UMAP(
    n_components=2,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
).fit_transform(embeddings)

df = pd.DataFrame(reduced_embeddings_2, columns=['x', 'y'])
df['title'] = titles
df['cluster'] = [str(c) for c in clusters]

cluster_df = df.loc[df.cluster != '-1', :]
outlier_df = df.loc[df.cluster == "-1", :]

plt.scatter(
    cluster_df['x'], cluster_df['y'],
    c=cluster_df['cluster'].astype(int),
    cmap='tab20b', s=2, alpha=0.6,
)
plt.scatter(outlier_df['x'], outlier_df['y'], s=10, alpha=0.05, c='grey')

plt.title('HDBSCAN Clustering of ArXiv NLP Abstracts (UMAP Reduced)')
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')  
plt.savefig("clusters.png", bbox_inches="tight")

> 📌 **本節重點**
> - `plt.scatter(x, y, c=...)`：位置看 `x, y`，顏色看 `c`。
> - 分群用 5 維、畫圖用 2 維：兩次 UMAP 各司其職。
> - `savefig` 把圖存成檔案；在 notebook 裡圖也會直接顯示。

## 10. 第四步：把零件組裝進 BERTopic（步驟 6）

```python
topic_model = BERTopic(
    embedding_model = embedding_model,
    umap_model = umap_model,
    hdbscan_model = hdbscan_model,
    verbose = True,
).fit(abstracts, embeddings)
```

**為什麼 `.fit()` 同時要 `abstracts` 和 `embeddings`？**
- `embeddings`：直接用來降維、分群，**不用重新嵌入**。
- `abstracts`（原始文字）：分群之後，把「同一群的所有摘要」合在一起算 **c-TF-IDF**，找出這個群最有代表性的詞 → 這就是主題的關鍵字。

**c-TF-IDF 是什麼？** 一般 TF-IDF 看「一篇文件」裡哪個詞重要；c-TF-IDF（class-based）把「**整個 cluster 當成一份大文件**」，找出「這一群常出現、其他群不常出現」的詞。

> ⚠️ 兩個觀察：
> 1. `umap_model`、`hdbscan_model` 前面已經 `fit` 過；BERTopic 的 `.fit()` 內部會**再各 `fit` 一次**。種子固定，結果通常相近，但多花一些時間。
> 2. BERTopic 會依主題大小**重新編號**（主題 0 通常最大），所以第 9 節圖上的 cluster 顏色編號，**不一定**等於這裡的 Topic 編號。

In [ ]:
#前面是手動學習BERT過程 但可以省略 直接把模型架好 送到BERT
#需要abstracts 跟embedding是因為 透過embedding 先做cluster
#然後從cluster裡的那些abstract做c-TF-IDF 做topic modeling
topic_model = BERTopic(
    embedding_model = embedding_model,
    umap_model = umap_model,
    hdbscan_model = hdbscan_model,
    verbose = True,
).fit(abstracts, embeddings)

> 📌 **本節重點**
> - BERTopic = 你的 `embedding_model` + `umap_model` + `hdbscan_model` + 它自己的 **c-TF-IDF**。
> - 傳 `embeddings` 跳過重新嵌入；傳 `abstracts` 是為了抽關鍵字。
> - 主題 `-1` 是離群文件，不是真正的主題。
> - （原程式把方法裡的 docstring 當成說明；攤平後我把它改成同文字的 `#` 註解，內容不變。）

## 11. 查看主題（步驟 7）

| 呼叫 | 作用 |
|---|---|
| `get_topic_info()` | 總表：主題編號、文章數、名稱、代表字（`Topic = -1` 是離群值） |
| `get_topic(0)` | 看**單一主題**的關鍵字與 c-TF-IDF 分數 |
| `find_topics("topic modeling")` | 用關鍵字或一句話**搜尋**最相關的主題與相似度 |

In [ ]:
print(topic_model.get_topic_info())                    #快速獲取找到的主題描述
print(topic_model.get_topic(0))                        #查看個別主題 探索最能代表這篇文章的關鍵字
print(topic_model.find_topics("topic modeling"))       #根據關鍵字找尋特定主題文章

> 📌 **本節重點**：先 `get_topic_info()` 看全貌，再用 `get_topic(n)` 深入單一主題；`find_topics` 是「由關鍵字找主題」。

## 12. 互動式視覺化（步驟 8）

BERTopic 的 `visualize_*()` 回傳的是 **Plotly 互動圖**，用 `write_html()` 存成網頁才能保留互動功能，再用瀏覽器開（如 `firefox topic_barchart.html`）。

| 圖 | 看什麼 |
|---|---|
| `visualize_documents` | 每篇文章一個點，依主題上色 |
| `visualize_barchart` | 各主題的關鍵字長條圖 |
| `visualize_heatmap(n_clusters=30)` | 主題之間的相似度熱圖 |
| `visualize_hierarchy` | 主題的階層關係 |

**`visualize_documents` 的參數對應**（我查了原始碼）：

| 你寫的 | 對應參數 | 說明 |
|---|---|---|
| `docs = titles` | `docs` | 官方說明是「fit 時用的文件」，但在這個函式裡它**只用來顯示滑鼠提示文字**；傳標題，滑鼠移上去就顯示短標題，而不是一大段摘要。長度與順序必須和 `abstracts` 一致 |
| `reduced_embeddings = reduced_embeddings_2` | `reduced_embeddings` | 要求 **2 維**座標；這裡用第 9 節的 2 維版本是正確的 |
| `hide_annotations = True` | `hide_annotations` | 不在圖上標示主題文字，圖比較乾淨 |
| `width = 1200` | `width` | 圖寬 |

In [ ]:
#BERTopic 的這些 visualize_*() 圖表通常是 Plotly 互動式圖表，不是單純的靜態圖片。
#把整張互動式 Plotly 圖存成一個html網頁 可以保留互動功能
#透過 firefox topic_barchart.html 即可打開
fig = topic_model.visualize_documents(
    docs = titles,
    reduced_embeddings = reduced_embeddings_2,
    hide_annotations = True,
    width = 1200,
)
fig.write_html("documents.html")

fig_bar = topic_model.visualize_barchart()
fig_heatmap = topic_model.visualize_heatmap(n_clusters=30)
fig_hierarchy = topic_model.visualize_hierarchy()

fig_bar.write_html("topic_barchart.html")
fig_heatmap.write_html("topic_heatmap.html")
fig_hierarchy.write_html("topic_hierarchy.html")

> 📌 **本節重點**
> - Plotly 圖 → `write_html()` 存成網頁，才能保留互動。
> - `visualize_documents` 要 **2 維**座標；用 `reduced_embeddings_2` 才對。
> - 用參數名稱（`docs = ...`）呼叫，比只靠位置更不容易搞錯。

## 13. 微調關鍵字（一）：先備份，再用 KeyBERTInspired（步驟 9、10）

**為什麼要先備份？** 我們想比較「改之前 vs 改之後」，所以改之前要存一份：

```python
original_topics = deepcopy(topic_model.topic_representations_)
```

- 直接指派只是讓兩個名字指向同一個物件；`deepcopy` 連巢狀的 list、tuple 都完整複製一份。
- 補充：我查了 0.17.4 的 `update_topics`，它是**重新指派**新字典，不是就地修改，所以這個版本不 `deepcopy` 也很可能沒事。但 `deepcopy` 是不依賴內部實作的保險習慣。
- **一定要在 `update_topics` 之前備份**，否則備份到的就是改過的版本。

**KeyBERTInspired 在做什麼？** c-TF-IDF 靠詞頻統計，有時挑到不貼切的詞。KeyBERTInspired：
1. 找出主題最有代表性的幾篇文件，取得它們的 embedding；
2. 對每個候選關鍵字也算 embedding；
3. 用 **cosine similarity** 比較兩者，**重新排名**。

In [ ]:
# 一定要在 update_topics 之前做，否則存到的就已經是改過的版本
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
representation_model = KeyBERTInspired()
topic_model.update_topics(
    abstracts,
    representation_model=representation_model
)
print(topic_difference(topic_model, original_topics))

> 📌 **本節重點**
> - `update_topics(docs, representation_model=...)` 只**重新計算關鍵字**，不用重新分群、不用重新嵌入。
> - 用 `topic_difference` 並排「Original」與「Updated」，一眼看出差異。

## 14. 微調關鍵字（二）：MaximalMarginalRelevance（步驟 11）

有時前 5 個關鍵字長得太像（如 `model`、`models`、`modeling`）。**MMR** 同時考慮：
- **相關性**：這個詞跟主題有多相關
- **多樣性**：這個詞跟「已挑過的詞」有多不一樣

`diversity=0.2`：數字越大越重視多樣性（0～1）。

> ⚠️ 兩次 `update_topics` 是**各自獨立**的：我看過原始碼，每次都從原始文件重新計算 c-TF-IDF，再套用你給的表示模型。所以這一節的結果是「c-TF-IDF + MMR」，**不是**接在 KeyBERT 後面。這也是兩次都拿 `original_topics` 當比較基準的原因。
>
> 另外：**呼叫 `update_topics` 時，參數名稱一定要拼成 `representation_model`**。變數名稱拼錯沒關係，但參數名稱拼錯會出現 `TypeError`。

In [ ]:
representation_model = MaximalMarginalRelevance(diversity=0.2)
topic_model.update_topics(
    abstracts,
    representation_model=representation_model,
)
print(topic_difference(topic_model, original_topics))

> 📌 **本節重點**
> - KeyBERTInspired：用 embedding 相似度**重新排名**。MMR：用**多樣性**減少重複意思的詞。
> - 兩者各自嘗試，看哪種讓主題更好讀；沒有標準答案。

## 15. 讓生成式 LLM 為主題取名字（步驟 12）

關鍵字有時還是不夠直觀。這一步請 **GPT** 讀「代表文件 + 關鍵字」，用一個短短的人話標籤描述主題。

> ⚠️ **執行前先確認**
> - 這一步會對**每個主題各呼叫一次 OpenAI API**（你的資料有一百多個主題），**會產生費用**。`gpt-4o-mini` 相對便宜，但第一次建議先確認 `.env` 設定正確。
> - 每個主題的代表文件內容會**送到 OpenAI**；如果摘要有敏感內容，要先考慮。
> - 在你原本的 `run()` 裡，這一步是被註解掉的；這份教材把它放成一般 cell，沒有金鑰就**跳過這一格**即可，不影響前面的內容。

**程式逐行說明**

| 程式 | 說明 |
|---|---|
| `load_dotenv()` | 把 `.env` 裡的 `OPENAI_API_KEY` 讀進環境變數 |
| `openai.OpenAI()` | 建立官方 client；它自己會讀取環境變數裡的金鑰，所以程式裡**不用出現金鑰** |
| `OpenAI(client, model="gpt-4o-mini", prompt=PROMPT, ...)` | BERTopic 的表示模型：用這個 client、這個模型、這個提示詞 |
| `exponential_backoff=True` | 遇到 `RateLimitError`（呼叫太頻繁）時自動等待後重試，等待時間逐次加長 |
| `chat=True` | 想表示使用聊天模型；**但在 0.17.4 沒有這個參數**，它被 `**kwargs` 吸收後沒有用途（不會報錯，也沒效果）。目前一律是聊天模式 |
| `update_topics(abstracts, representation_model=...)` | 為每個主題挑代表文件、填提示詞、呼叫 GPT，把回傳的標籤當作新的主題表示 |

**BERTopic 內部做了什麼**（原始碼）：對每個主題 → 取 4 篇代表文件 → 用 `[DOCUMENTS]`、`[KEYWORDS]` 填提示詞 → 送出（含 system 提示詞 "You are an assistant that extracts high-level topics from texts."）→ 取回答 → 去掉開頭的 `topic: ` → 當成這個主題的標籤。

**要留意的細節**
- 輸出會在第一個換行處被截斷（BERTopic 預設 `stop` 是 `"\n"`），所以回答格式請維持一行。
- 回傳的每個主題只有一個 `(標籤, 1)`，所以 `topic_difference` 的 "Updated" 欄只會顯示標籤，不會有 5 個關鍵字。
- 想檢查實際送出的提示詞：跑完後 `representation_model.prompts_[0]` 就是第一個主題的完整提示詞；若裡面還看到 `[DOCUMENTS]`，就表示沒替換成功。

In [ ]:
load_dotenv()
client = openai.OpenAI()
representation_model = OpenAI(
    client,
    model="gpt-4o-mini",
    prompt=PROMPT,
    exponential_backoff=True,
    chat=True,
)
topic_model.update_topics(
    abstracts,
    representation_model=representation_model,
)
print(topic_difference(topic_model, original_topics))

> 📌 **本節重點**
> - 佔位符由 BERTopic 自動填入，**名字要用它規定的**：`[DOCUMENTS]`、`[KEYWORDS]`。
> - 金鑰放 `.env` + `load_dotenv()`，**不要寫進程式或提交到 git**。
> - 這是**逐主題呼叫 API**：主題多 → 呼叫多 → 花費高。先小規模試。
> - 回答格式要在提示詞裡寫清楚（`topic: <label>`），因為程式要靠格式解析。

## 16. 全課重點整理

### 流程回顧

| 步驟 | 工具 | 輸入 → 輸出 |
|---|---|---|
| ① 嵌入 | `SentenceTransformer` | 摘要 → 384 維向量 |
| ② 降維 | `UMAP` | 384 維 → 5 維（分群）／2 維（畫圖） |
| ③ 分群 | `HDBSCAN` | 低維向量 → cluster 編號（`-1` = 離群） |
| ④ 抽關鍵字 | BERTopic 內的 c-TF-IDF | 每群摘要 → 主題關鍵字 |
| ⑤ 視覺化 | Plotly + `write_html` | 主題模型 → 互動網頁 |
| ⑥ 微調 | `KeyBERTInspired`／`MaximalMarginalRelevance` | 關鍵字 → 更好讀的關鍵字 |
| ⑦ 生成標題 | `OpenAI`（GPT） | 代表文件 + 關鍵字 → 一句話標籤 |

### 必記的 7 件事

1. 主題建模是**非監督式**的：不需要標籤。
2. BERTopic 是**可抽換零件**的流水線。
3. UMAP 保留相近關係並降維；**分群用 5 維、畫圖用 2 維**。
4. HDBSCAN 不用指定群數，雜訊標成 `-1`；`-1` 不是主題。
5. `.fit(docs, embeddings)` 兩個都要給。
6. 比較前後差異：**先備份，再 `update_topics`**；`update_topics` 的參數叫 `representation_model`。
7. 佔位符是**工具規定的**（BERTopic：`[DOCUMENTS]`、`[KEYWORDS]`）；API 金鑰放 `.env`、不進 git。

### 常見錯誤

| 現象 | 可能原因 |
|---|---|
| `AttributeError: ... 'abstract'` | 把 `abstracts` 少打了 s |
| `TypeError: unexpected keyword argument` | `update_topics` 的參數名稱拼錯（要 `representation_model`） |
| GPT 的標籤前面帶著 `topic : ` | 提示詞格式寫成 `topic :`，程式只去掉 `topic: ` |
| 提示詞裡的 `[DOCUMENTS]` 沒被換 | 佔位符名字或大小寫不對（不會報錯） |
| `AuthenticationError` | 沒有 `.env`、金鑰錯誤，或沒呼叫 `load_dotenv()` |
| 圖上的顏色編號與 Topic 編號對不上 | BERTopic 依主題大小重新編號 |
| `topic_difference` 報 `KeyError` | 主題數少於 `nr_topics` |

### 動手練習

1. 把 `min_cluster_size` 改成 20、100，比較群數與離群值比例。
2. 把 MMR 的 `diversity` 改成 0.5、0.8，看關鍵字怎麼變。
3. 修改 `PROMPT`：改成「請用繁體中文，最多 5 個字」，觀察標籤的變化（注意仍保持 `topic: ` 開頭）。
4. 印出 `representation_model.prompts_[0]`，確認佔位符真的被換掉。
5. 進階：先用 `topics` 參數或縮小資料，只對少數主題呼叫 GPT，降低費用。

### 延伸方向

- BERTopic 官方文件 *Representation Models*：還有 `TextGeneration`（本機模型）、`Cohere`、`LiteLLM` 等
- 換其他 embedding 模型 / 降維 / 分群方法對結果的影響
- 用 LLM 評估主題品質，或自動合併相似主題